# 从稳定 BCE 到训练诊断：Banknote Logistic Regression

这份共享 Notebook 用同一份本地 UCI Banknote 快照连接“数值优化”和“训练诊断”。先验证数据与划分，再手写稳定 BCE、梯度、Armijo 和停止状态机，最后只在端点与 scikit-learn 比较。五条训练轨迹是唯一的真实数据运行；极端 logit 与失败条件是独立探针，不会伪装成第六条运行。

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import os
import sys

import numpy as np
import pandas
import scipy
from scipy.special import expit
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, roc_auc_score

CONTRACT_VERSION = "numerical-methods-batch-4-v1"
EXPECTED_DATASET_SHA256 = "04c25b8d1ffaada3c392682185a94b089b7fd214ac82eabeb07faaa3d9a1efaf"
EXPECTED_SCHEMA = ["banknote_id", "variance", "skewness", "curtosis", "entropy", "class", "split"]
FEATURES = ["variance", "skewness", "curtosis", "entropy"]
PARAMETER_ORDER = [*FEATURES, "intercept"]

def locate_dataset():
    candidates = [
        Path("banknote-authentication.csv"),
        Path(os.environ["ML_ATLAS_BANKNOTE_DATA_PATH"]) if os.environ.get("ML_ATLAS_BANKNOTE_DATA_PATH") else None,
        Path("public/datasets/numerical-methods/banknote-authentication.csv"),
    ]
    for candidate in candidates:
        if candidate is not None and candidate.is_file():
            return candidate
    raise FileNotFoundError("Place banknote-authentication.csv beside this Notebook or set ML_ATLAS_BANKNOTE_DATA_PATH")

dataset_path = locate_dataset()
observed_dataset_sha256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
assert observed_dataset_sha256 == EXPECTED_DATASET_SHA256

# Pandas owns local loading. Schema, row ids, labels, split counts, and finiteness
# are validated before any feature or target array is converted to NumPy.
frame = pandas.read_csv(dataset_path)
assert list(frame.columns) == EXPECTED_SCHEMA
assert frame.shape == (1372, 7)
assert frame["banknote_id"].tolist() == list(range(1, 1373))
assert not frame.duplicated(subset=["banknote_id"]).any()
assert not frame.isna().any().any()
assert set(frame["class"].tolist()) == {0, 1}
assert set(frame["split"].tolist()) == {"train", "validation", "test"}
assert frame["split"].value_counts().to_dict() == {"train": 960, "validation": 206, "test": 206}
assert {
    split: frame.loc[frame["split"] == split, "class"].value_counts().sort_index().to_dict()
    for split in ("train", "validation", "test")
} == {
    "train": {0: 533, 1: 427},
    "validation": {0: 115, 1: 91},
    "test": {0: 114, 1: 92},
}
assert np.isfinite(frame[FEATURES].to_numpy(dtype=np.float64)).all()

split_frames = {name: frame.loc[frame["split"] == name] for name in ("train", "validation", "test")}
X_raw = {name: part[FEATURES].to_numpy(dtype=np.float64) for name, part in split_frames.items()}
y = {name: part["class"].to_numpy(dtype=np.float64) for name, part in split_frames.items()}
loader_record = {
    "library": "pandas",
    "call": "pandas.read_csv",
    "datasetPublicPath": "/datasets/numerical-methods/banknote-authentication.csv",
    "datasetSha256": observed_dataset_sha256,
    "schema": EXPECTED_SCHEMA,
    "schemaValidatedBeforeNumpy": True,
    "rowCount": int(len(frame)),
    "splitCounts": {name: int(len(part)) for name, part in split_frames.items()},
}
print(json.dumps({"loader": loader_record, "python": sys.version.split()[0]}, ensure_ascii=False, sort_keys=True))

{"loader": {"call": "pandas.read_csv", "datasetPublicPath": "/datasets/numerical-methods/banknote-authentication.csv", "datasetSha256": "04c25b8d1ffaada3c392682185a94b089b7fd214ac82eabeb07faaa3d9a1efaf", "library": "pandas", "rowCount": 1372, "schema": ["banknote_id", "variance", "skewness", "curtosis", "entropy", "class", "split"], "schemaValidatedBeforeNumpy": true, "splitCounts": {"test": 206, "train": 960, "validation": 206}}, "python": "3.12.13"}


## 1. 只用训练集拟合尺度

固定划分已经写入 CSV。均值与总体标准差只从 960 条训练记录计算，验证集和测试集只复用这些量。这样缩放改变的是优化条件，而不是偷看留出数据。

In [2]:
train_mean = X_raw["train"].mean(axis=0)
train_scale = X_raw["train"].std(axis=0, ddof=0)
assert np.all(train_scale > 0)
X_standardized = {name: (matrix - train_mean) / train_scale for name, matrix in X_raw.items()}

locked_mean = np.array([0.46886307781249986, 1.9775978456250036, 1.3202396866562518, -1.1418097847916664])
locked_scale = np.array([2.8049705227712813, 5.81400805653475, 4.234924404032209, 2.0726581960156034])
# The normalized decimal CSV can round-trip a few ulps away from the raw-source
# statistics recorded in its manifest. This is far tighter than the 1e-9
# cross-runtime scalar contract while avoiding a parser-specific equality test.
assert np.allclose(train_mean, locked_mean, atol=1e-12, rtol=0)
assert np.allclose(train_scale, locked_scale, atol=1e-12, rtol=0)
preprocessing_record = {
    "fitSplit": "train",
    "ddof": 0,
    "features": FEATURES,
    "trainMeans": {name: float(value) for name, value in zip(FEATURES, train_mean, strict=True)},
    "trainScales": {name: float(value) for name, value in zip(FEATURES, train_scale, strict=True)},
}
print(json.dumps(preprocessing_record, ensure_ascii=False, sort_keys=True))

{"ddof": 0, "features": ["variance", "skewness", "curtosis", "entropy"], "fitSplit": "train", "trainMeans": {"curtosis": 1.32023968665625, "entropy": -1.1418097847916668, "skewness": 1.9775978456249999, "variance": 0.46886307781250003}, "trainScales": {"curtosis": 4.234924404032209, "entropy": 2.072658196015603, "skewness": 5.81400805653475, "variance": 2.8049705227712796}}


## 2. 先让 BCE 在极端 logit 下仍然可信

概率域写法会在 $p=0$ 或 $p=1$ 处遇到 $\log 0$。权威实现直接计算 $\operatorname{logaddexp}(0,z)-yz$；naive 写法只留在隔离的比较探针里，非有限值绝不进入 JSON。

In [3]:
def stable_bce(logits, targets):
    logits = np.asarray(logits, dtype=np.float64)
    targets = np.asarray(targets, dtype=np.float64)
    return float(np.mean(np.logaddexp(0.0, logits) - targets * logits))

stable_wrong_positive = stable_bce(np.array([1000.0]), np.array([0.0]))
stable_wrong_negative = stable_bce(np.array([-1000.0]), np.array([1.0]))
stable_correct_positive = stable_bce(np.array([1000.0]), np.array([1.0]))
stable_correct_negative = stable_bce(np.array([-1000.0]), np.array([0.0]))
with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    naive_probabilities = expit(np.array([1000.0, -1000.0, 1000.0, -1000.0]))
    naive_targets = np.array([0.0, 1.0, 1.0, 0.0])
    naive_values = -(naive_targets * np.log(naive_probabilities) + (1.0 - naive_targets) * np.log(1.0 - naive_probabilities))

assert stable_wrong_positive == 1000.0 and stable_wrong_negative == 1000.0
assert stable_correct_positive == 0.0 and stable_correct_negative == 0.0
assert not np.isfinite(naive_values).all()
assert np.allclose(expit(np.array([-1000.0, 0.0, 1000.0])), np.array([0.0, 0.5, 1.0]))
extreme_logit_check = {
    "probeLogits": [1000.0, -1000.0],
    "naiveFinite": False,
    "naiveOutcomeKinds": ["positive-infinity", "positive-infinity", "not-a-number", "not-a-number"],
    "stableWrongPositive": stable_wrong_positive,
    "stableWrongNegative": stable_wrong_negative,
    "stableCorrectPositive": stable_correct_positive,
    "stableCorrectNegative": stable_correct_negative,
    "scipyExpitAgreement": True,
}
print(json.dumps(extreme_logit_check, ensure_ascii=False, sort_keys=True))

{"naiveFinite": false, "naiveOutcomeKinds": ["positive-infinity", "positive-infinity", "not-a-number", "not-a-number"], "probeLogits": [1000.0, -1000.0], "scipyExpitAgreement": true, "stableCorrectNegative": 0.0, "stableCorrectPositive": 0.0, "stableWrongNegative": 1000.0, "stableWrongPositive": 1000.0}


## 3. 同一个目标函数同时返回损失和梯度

训练目标在平均 BCE 上加入 $\lambda\lVert w\rVert_2^2/2$，但不惩罚截距。验证与测试只报告数据 BCE。下一单元用中心差分独立检查五个偏导。

In [4]:
L2 = 1e-3

def loss_and_grad(X, targets, parameters, l2=L2):
    parameters = np.asarray(parameters, dtype=np.float64)
    weights, intercept = parameters[:-1], parameters[-1]
    logits = X @ weights + intercept
    probabilities = expit(logits)
    residual = probabilities - targets
    bce = stable_bce(logits, targets)
    objective = bce + 0.5 * l2 * float(weights @ weights)
    gradient = np.r_[X.T @ residual / len(targets) + l2 * weights, residual.mean()]
    return {"objective": float(objective), "bce": float(bce), "gradient": gradient}

def validation_bce(X, targets, parameters):
    parameters = np.asarray(parameters, dtype=np.float64)
    return stable_bce(X @ parameters[:-1] + parameters[-1], targets)

In [5]:
probe_parameters = np.array([0.2, -0.1, 0.05, 0.15, -0.3], dtype=np.float64)
probe_step = 1e-6
analytic_probe = loss_and_grad(X_standardized["train"], y["train"], probe_parameters)
numeric_gradient = []
for index in range(len(probe_parameters)):
    delta = np.zeros_like(probe_parameters)
    delta[index] = probe_step
    plus = loss_and_grad(X_standardized["train"], y["train"], probe_parameters + delta)["objective"]
    minus = loss_and_grad(X_standardized["train"], y["train"], probe_parameters - delta)["objective"]
    numeric_gradient.append((plus - minus) / (2.0 * probe_step))
numeric_gradient = np.asarray(numeric_gradient)
gradient_error = np.abs(numeric_gradient - analytic_probe["gradient"])
max_gradient_error = float(gradient_error.max())
assert max_gradient_error <= 2e-9

unregularized_probe = loss_and_grad(X_standardized["train"], y["train"], probe_parameters, l2=0.0)
regularization_delta = analytic_probe["gradient"] - unregularized_probe["gradient"]
assert np.allclose(regularization_delta[:-1], L2 * probe_parameters[:-1], atol=1e-15, rtol=0)
assert regularization_delta[-1] == 0.0
gradient_check = {
    "probeParameters": probe_parameters.tolist(),
    "step": probe_step,
    "analytic": analytic_probe["gradient"].tolist(),
    "centeredDifference": numeric_gradient.tolist(),
    "absoluteErrors": gradient_error.tolist(),
    "maxAbsoluteError": max_gradient_error,
    "interceptExcludedFromL2": True,
}
print(json.dumps({"maxAbsoluteError": max_gradient_error, "interceptExcludedFromL2": True}, sort_keys=True))

{"interceptExcludedFromL2": true, "maxAbsoluteError": 8.32334340339358e-11}


## 4. Armijo 只接受训练目标的充分下降

方向固定为负梯度。每次尝试只用训练目标判断是否充分下降；验证损失只在接受后更新模型选择检查点。停止优先级固定为梯度范数、损失与参数步长的合取、验证耐心，最后才是最大迭代数。

In [6]:
GRADIENT_TOLERANCE = 1e-5
RELATIVE_LOSS_TOLERANCE = 1e-10
PARAMETER_STEP_TOLERANCE = 1e-7
VALIDATION_MIN_DELTA = 1e-7
VALIDATION_PATIENCE = 60
MAX_ITERATIONS = 500

def terminal(kind, reason, iteration, attempted_iteration=None):
    value = {
        "kind": kind,
        "reason": reason,
        "iteration": int(iteration),
        "messageKey": f"batch4.terminal.{reason}",
    }
    if attempted_iteration is not None:
        value["attemptedIteration"] = int(attempted_iteration)
    return value

def armijo_step(X, targets, parameters, current, initial_step=32.0, c=1e-4, rho=0.5, max_backtracks=30, minimum_step=1e-12):
    gradient = current["gradient"]
    gradient_norm_squared = float(gradient @ gradient)
    for backtracks in range(max_backtracks + 1):
        alpha = float(initial_step * rho ** backtracks)
        if alpha < minimum_step:
            break
        candidate = parameters - alpha * gradient
        if not np.isfinite(candidate).all():
            continue
        candidate_state = loss_and_grad(X, targets, candidate)
        if not np.isfinite(candidate_state["objective"]) or not np.isfinite(candidate_state["gradient"]).all():
            continue
        right_hand_side = current["objective"] - c * alpha * gradient_norm_squared
        if candidate_state["objective"] <= right_hand_side:
            return {
                "accepted": True,
                "parameters": candidate,
                "state": candidate_state,
                "acceptedStepSize": alpha,
                "backtrackCount": backtracks,
                "armijoRightHandSide": float(right_hand_side),
            }
    return {"accepted": False, "reason": "line-search-failed"}

def should_stop(iteration, gradient_norm, relative_objective_change, parameter_step_norm, best_iteration, max_iterations=MAX_ITERATIONS):
    if gradient_norm <= GRADIENT_TOLERANCE:
        return terminal("mathematical-convergence", "gradient-norm", iteration)
    if relative_objective_change <= RELATIVE_LOSS_TOLERANCE and parameter_step_norm <= PARAMETER_STEP_TOLERANCE:
        return terminal("mathematical-convergence", "loss-and-step", iteration)
    if iteration - best_iteration >= VALIDATION_PATIENCE:
        return terminal("model-selection", "validation-patience", iteration)
    if iteration >= max_iterations:
        return terminal("safety", "max-iterations", iteration)
    return None

In [7]:
def trace_point(iteration, state, validation_loss, parameter_step_norm, accepted_step_size, backtrack_count, relative_change, is_best, parameters):
    return {
        "iteration": int(iteration),
        "trainBce": float(state["bce"]),
        "validationBce": float(validation_loss),
        "objective": float(state["objective"]),
        "gradientNorm": float(np.linalg.norm(state["gradient"])),
        "parameterStepNorm": float(parameter_step_norm),
        "acceptedStepSize": float(accepted_step_size),
        "backtrackCount": int(backtrack_count),
        "relativeObjectiveChange": None if relative_change is None else float(relative_change),
        "isBestValidation": bool(is_best),
        "parameters": np.asarray(parameters, dtype=np.float64).tolist(),
    }

def train_logistic(run_id, X_train, y_train, X_validation, y_validation, feature_space, method, step, max_iterations=MAX_ITERATIONS, armijo_max_backtracks=30):
    parameters = np.zeros(5, dtype=np.float64)
    current = loss_and_grad(X_train, y_train, parameters)
    current_validation = validation_bce(X_validation, y_validation, parameters)
    start = trace_point(0, current, current_validation, 0.0, 0.0, 0, None, True, parameters)
    trace = [start]
    best = {"iteration": 0, "bce": float(current_validation), "parameters": parameters.tolist()}
    first_backtrack = None
    terminal_state = None

    for iteration in range(1, max_iterations + 1):
        if method == "armijo":
            proposal = armijo_step(
                X_train, y_train, parameters, current,
                initial_step=step, max_backtracks=armijo_max_backtracks,
            )
            if not proposal["accepted"]:
                terminal_state = terminal("safety", "line-search-failed", iteration - 1, iteration)
                break
            candidate = proposal["parameters"]
            candidate_state = proposal["state"]
            accepted_step = proposal["acceptedStepSize"]
            backtracks = proposal["backtrackCount"]
        else:
            candidate = parameters - step * current["gradient"]
            if not np.isfinite(candidate).all():
                terminal_state = terminal("safety", "non-finite", iteration - 1, iteration)
                break
            # The explicit non-finite probe must terminate safely without
            # leaking parser-specific warnings or temporary kernel paths into
            # the published Notebook output.
            with np.errstate(over="ignore", invalid="ignore"):
                candidate_state = loss_and_grad(X_train, y_train, candidate)
            if not np.isfinite(candidate_state["objective"]) or not np.isfinite(candidate_state["gradient"]).all():
                terminal_state = terminal("safety", "non-finite", iteration - 1, iteration)
                break
            accepted_step = float(step)
            backtracks = 0

        candidate_validation = validation_bce(X_validation, y_validation, candidate)
        if not np.isfinite(candidate_validation):
            terminal_state = terminal("safety", "non-finite", iteration - 1, iteration)
            break
        parameter_step_norm = float(np.linalg.norm(candidate - parameters))
        relative_change = abs(candidate_state["objective"] - current["objective"]) / max(1.0, abs(current["objective"]))
        is_best = candidate_validation < best["bce"] - VALIDATION_MIN_DELTA
        if is_best:
            best = {"iteration": iteration, "bce": float(candidate_validation), "parameters": candidate.tolist()}
        point = trace_point(
            iteration, candidate_state, candidate_validation, parameter_step_norm,
            accepted_step, backtracks, relative_change, is_best, candidate,
        )
        trace.append(point)
        if first_backtrack is None and backtracks > 0:
            first_backtrack = point.copy()

        parameters = candidate
        current = candidate_state
        terminal_state = should_stop(
            iteration, point["gradientNorm"], relative_change,
            parameter_step_norm, best["iteration"], max_iterations,
        )
        if terminal_state is not None:
            break

    if terminal_state is None:
        raise RuntimeError(f"{run_id} ended without a terminal state")
    if terminal_state["iteration"] != trace[-1]["iteration"]:
        raise RuntimeError(f"{run_id} terminal does not retain the last finite trace row")
    eligible = terminal_state["kind"] == "mathematical-convergence"
    return {
        "runId": run_id,
        "featureSpace": feature_space,
        "method": method,
        "config": {
            "l2": L2,
            "step": float(step),
            "maxIterations": int(max_iterations),
            "gradientTolerance": GRADIENT_TOLERANCE,
            "relativeLossTolerance": RELATIVE_LOSS_TOLERANCE,
            "parameterStepTolerance": PARAMETER_STEP_TOLERANCE,
            "validationMinDelta": VALIDATION_MIN_DELTA,
            "validationPatience": VALIDATION_PATIENCE,
            "armijo": None if method == "fixed" else {
                "initialStep": float(step), "c": 1e-4, "rho": 0.5,
                "maxBacktracks": int(armijo_max_backtracks), "minimumStep": 1e-12,
            },
        },
        "start": start,
        "firstBacktrack": first_backtrack,
        "bestValidation": best,
        "terminal": terminal_state,
        "eligibleForFinalSelection": eligible,
        "trace": trace,
    }

## 5. 五条固定运行：尺度、步长与线搜索

五条运行都从零参数开始，并按持久化的训练行顺序做全批量更新。raw 与 standardized 的 L2 几何并不相同，因此这里比较的是早期轨迹和可用步长，不把最终 BCE 当成单纯的模型优劣排名。

In [8]:
RUN_ORDER = ["raw-fixed", "standardized-too-small", "standardized-stable", "standardized-too-large", "standardized-armijo"]
run_specs = [
    ("raw-fixed", "raw", "fixed", 4.0),
    ("standardized-too-small", "standardized", "fixed", 0.02),
    ("standardized-stable", "standardized", "fixed", 4.0),
    ("standardized-too-large", "standardized", "fixed", 32.0),
    ("standardized-armijo", "standardized", "armijo", 32.0),
]
runs = []
for run_id, feature_space, method, step in run_specs:
    matrices = X_raw if feature_space == "raw" else X_standardized
    runs.append(train_logistic(
        run_id, matrices["train"], y["train"], matrices["validation"], y["validation"],
        feature_space, method, step,
    ))

run_by_id = {run["runId"]: run for run in runs}
locked_terminals = {
    "raw-fixed": ("validation-patience", 112, 52),
    "standardized-too-small": ("max-iterations", 500, 500),
    "standardized-stable": ("gradient-norm", 484, 484),
    "standardized-too-large": ("validation-patience", 73, 13),
    "standardized-armijo": ("gradient-norm", 48, 48),
}
for run_id, (reason, terminal_iteration, best_iteration) in locked_terminals.items():
    run = run_by_id[run_id]
    assert run["terminal"]["reason"] == reason
    assert run["terminal"]["iteration"] == terminal_iteration
    assert run["bestValidation"]["iteration"] == best_iteration
assert run_by_id["standardized-armijo"]["firstBacktrack"]["iteration"] == 1
assert run_by_id["standardized-armijo"]["firstBacktrack"]["backtrackCount"] == 1
assert run_by_id["standardized-armijo"]["firstBacktrack"]["acceptedStepSize"] == 16.0

armijo_run = run_by_id["standardized-armijo"]
armijo_sufficient_decrease = []
for previous, current_point in zip(armijo_run["trace"], armijo_run["trace"][1:]):
    right_hand_side = previous["objective"] - 1e-4 * current_point["acceptedStepSize"] * previous["gradientNorm"] ** 2
    armijo_sufficient_decrease.append(current_point["objective"] <= right_hand_side + 1e-15)
assert all(armijo_sufficient_decrease)
armijo_check = {
    "initialTrialStep": 32.0,
    "initialTrialAccepted": False,
    "firstAcceptedStep": 16.0,
    "firstBacktrackCount": 1,
    "allAcceptedRowsSatisfySufficientDecrease": True,
}
print(json.dumps({run["runId"]: {"terminal": run["terminal"], "best": run["bestValidation"]["iteration"], "rows": len(run["trace"])} for run in runs}, sort_keys=True))

{"raw-fixed": {"best": 52, "rows": 113, "terminal": {"iteration": 112, "kind": "model-selection", "messageKey": "batch4.terminal.validation-patience", "reason": "validation-patience"}}, "standardized-armijo": {"best": 48, "rows": 49, "terminal": {"iteration": 48, "kind": "mathematical-convergence", "messageKey": "batch4.terminal.gradient-norm", "reason": "gradient-norm"}}, "standardized-stable": {"best": 484, "rows": 485, "terminal": {"iteration": 484, "kind": "mathematical-convergence", "messageKey": "batch4.terminal.gradient-norm", "reason": "gradient-norm"}}, "standardized-too-large": {"best": 13, "rows": 74, "terminal": {"iteration": 73, "kind": "model-selection", "messageKey": "batch4.terminal.validation-patience", "reason": "validation-patience"}}, "standardized-too-small": {"best": 500, "rows": 501, "terminal": {"iteration": 500, "kind": "safety", "messageKey": "batch4.terminal.max-iterations", "reason": "max-iterations"}}}


In [9]:
priority_fixtures = [
    should_stop(3, 1e-6, 1e-12, 1e-8, 0, 3),
    should_stop(3, 1e-4, 1e-12, 1e-8, 0, 3),
    should_stop(60, 1e-4, 1e-4, 1e-3, 0, 60),
    should_stop(5, 1e-4, 1e-4, 1e-3, 4, 5),
]
assert [item["reason"] for item in priority_fixtures] == [
    "gradient-norm", "loss-and-step", "validation-patience", "max-iterations",
]

line_search_probe = train_logistic(
    "line-search-probe", X_standardized["train"], y["train"],
    X_standardized["validation"], y["validation"], "standardized", "armijo", 32.0,
    max_iterations=10, armijo_max_backtracks=0,
)
non_finite_probe = train_logistic(
    "non-finite-probe", X_standardized["train"], y["train"],
    X_standardized["validation"], y["validation"], "standardized", "fixed", sys.float_info.max,
    max_iterations=10,
)
assert line_search_probe["terminal"] == terminal("safety", "line-search-failed", 0, 1)
assert non_finite_probe["terminal"] == terminal("safety", "non-finite", 0, 1)
assert len(line_search_probe["trace"]) == 1 and len(non_finite_probe["trace"]) == 1
terminal_fixtures = [
    {"fixtureId": "priority-gradient", "terminal": priority_fixtures[0]},
    {"fixtureId": "priority-loss-step", "terminal": priority_fixtures[1]},
    {"fixtureId": "priority-validation", "terminal": priority_fixtures[2]},
    {"fixtureId": "priority-max", "terminal": priority_fixtures[3]},
    {"fixtureId": "non-finite-probe", "terminal": non_finite_probe["terminal"], "lastFinite": non_finite_probe["trace"][-1]},
    {"fixtureId": "line-search-probe", "terminal": line_search_probe["terminal"], "lastFinite": line_search_probe["trace"][-1]},
]
print(json.dumps({row["fixtureId"]: row["terminal"] for row in terminal_fixtures}, sort_keys=True))

{"line-search-probe": {"attemptedIteration": 1, "iteration": 0, "kind": "safety", "messageKey": "batch4.terminal.line-search-failed", "reason": "line-search-failed"}, "non-finite-probe": {"attemptedIteration": 1, "iteration": 0, "kind": "safety", "messageKey": "batch4.terminal.non-finite", "reason": "non-finite"}, "priority-gradient": {"iteration": 3, "kind": "mathematical-convergence", "messageKey": "batch4.terminal.gradient-norm", "reason": "gradient-norm"}, "priority-loss-step": {"iteration": 3, "kind": "mathematical-convergence", "messageKey": "batch4.terminal.loss-and-step", "reason": "loss-and-step"}, "priority-max": {"iteration": 5, "kind": "safety", "messageKey": "batch4.terminal.max-iterations", "reason": "max-iterations"}, "priority-validation": {"iteration": 60, "kind": "model-selection", "messageKey": "batch4.terminal.validation-patience", "reason": "validation-patience"}}


## 6. 先按数值资格选模型，再做一次库端点核对

只有数学收敛的运行能进入最终选择；这排除了 `standardized-too-large` 的短暂低验证损失。选中的 Armijo 最佳检查点在阈值 0.5 下报告测试 BCE、准确率、基于概率的 ROC-AUC 和混淆矩阵。LBFGS 只核对最终概率与参数方向，17 次库迭代不会与手写轨迹逐步对齐。

In [10]:
eligible_runs = [run for run in runs if run["eligibleForFinalSelection"]]
selected_run = min(eligible_runs, key=lambda run: run["bestValidation"]["bce"])
assert selected_run["runId"] == "standardized-armijo"
assert run_by_id["standardized-too-large"]["bestValidation"]["bce"] < selected_run["bestValidation"]["bce"]
selected_parameters = np.asarray(selected_run["bestValidation"]["parameters"], dtype=np.float64)

manual_test_logits = X_standardized["test"] @ selected_parameters[:-1] + selected_parameters[-1]
manual_test_probabilities = expit(manual_test_logits)
manual_predictions = (manual_test_probabilities >= 0.5).astype(np.int64)
manual_metrics = {
    "testBce": stable_bce(manual_test_logits, y["test"]),
    "accuracy": float(accuracy_score(y["test"], manual_predictions)),
    "rocAuc": float(roc_auc_score(y["test"], manual_test_probabilities)),
    "confusionMatrix": confusion_matrix(y["test"], manual_predictions, labels=[0, 1]).tolist(),
}

baseline = LogisticRegression(
    C=25 / 24,
    l1_ratio=0.0,
    solver="lbfgs",
    fit_intercept=True,
    tol=1e-12,
    max_iter=5000,
)
baseline.fit(X_standardized["train"], y["train"].astype(np.int64))
baseline_probabilities = baseline.predict_proba(X_standardized["test"])[:, 1]
baseline_predictions = (baseline_probabilities >= 0.5).astype(np.int64)
baseline_parameters = np.r_[baseline.coef_[0], baseline.intercept_[0]]
baseline_metrics = {
    "testBce": float(log_loss(y["test"], baseline_probabilities, labels=[0, 1])),
    "accuracy": float(accuracy_score(y["test"], baseline_predictions)),
    "rocAuc": float(roc_auc_score(y["test"], baseline_probabilities)),
    "confusionMatrix": confusion_matrix(y["test"], baseline_predictions, labels=[0, 1]).tolist(),
}
probability_difference = np.abs(manual_test_probabilities - baseline_probabilities)
coefficient_cosine = float(
    selected_parameters[:-1] @ baseline_parameters[:-1]
    / (np.linalg.norm(selected_parameters[:-1]) * np.linalg.norm(baseline_parameters[:-1]))
)
baseline_comparison = {
    "predictionAgreement": float(np.mean(manual_predictions == baseline_predictions)),
    "maxProbabilityDifference": float(probability_difference.max()),
    "meanProbabilityDifference": float(probability_difference.mean()),
    "coefficientDirectionCosine": coefficient_cosine,
    "endpointOnly": True,
    "perIterationComparison": False,
}
final_report = {
    "runId": selected_run["runId"],
    "checkpointIteration": selected_run["bestValidation"]["iteration"],
    "threshold": 0.5,
    "rocAucInput": "probabilities",
    "manual": manual_metrics,
}
baseline_report = {
    "library": "scikit-learn",
    "version": sklearn.__version__,
    "class": "LogisticRegression",
    "config": {"C": 25 / 24, "l1_ratio": 0.0, "solver": "lbfgs", "fit_intercept": True, "tol": 1e-12, "max_iter": 5000},
    "trainingRows": 960,
    "reportedIterations": int(baseline.n_iter_[0]),
    "parameters": baseline_parameters.tolist(),
    "metrics": baseline_metrics,
    "endpointOnly": True,
    "perIterationComparison": False,
}

assert abs(manual_metrics["testBce"] - 0.0551101232) <= 1e-10
assert abs(baseline_metrics["testBce"] - 0.0550980756) <= 1e-10
assert manual_metrics["confusionMatrix"] == [[110, 4], [0, 92]]
assert baseline_metrics["confusionMatrix"] == [[110, 4], [0, 92]]
assert baseline_report["reportedIterations"] == 17
print(json.dumps({"selectedRunId": selected_run["runId"], "manual": manual_metrics, "baseline": baseline_metrics, "comparison": baseline_comparison}, sort_keys=True))

{"baseline": {"accuracy": 0.9805825242718447, "confusionMatrix": [[110, 4], [0, 92]], "rocAuc": 0.9994279176201373, "testBce": 0.05509807557568522}, "comparison": {"coefficientDirectionCosine": 0.9999999991335304, "endpointOnly": true, "maxProbabilityDifference": 0.00015086178291673358, "meanProbabilityDifference": 1.2517068894184337e-05, "perIterationComparison": false, "predictionAgreement": 1.0}, "manual": {"accuracy": 0.9805825242718447, "confusionMatrix": [[110, 4], [0, 92]], "rocAuc": 0.9994279176201373, "testBce": 0.055110123229490826}, "selectedRunId": "standardized-armijo"}


In [11]:
output_dir = Path(os.environ.get("ML_ATLAS_NUMERICAL_BATCH4_OUTPUT_DIR", "batch-4-outputs"))
output_dir.mkdir(parents=True, exist_ok=True)

def write_json(name, value):
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

run_summaries = {
    run["runId"]: {
        "runId": run["runId"],
        "featureSpace": run["featureSpace"],
        "method": run["method"],
        "config": run["config"],
        "start": run["start"],
        "firstBacktrack": run["firstBacktrack"],
        "bestValidation": run["bestValidation"],
        "terminal": run["terminal"],
        "eligibleForFinalSelection": run["eligibleForFinalSelection"],
        "traceRowCount": len(run["trace"]),
    }
    for run in runs
}

optimization_summary = {
    "contractVersion": CONTRACT_VERSION,
    "outputId": "banknote-logistic-optimization-summary",
    "datasetSha256": observed_dataset_sha256,
    "constantsSha256": "f759186d0be526e84c80f6930d91981af0d303e8138edcbae4f65e4f3d30f041",
    "loader": loader_record,
    "preprocessing": preprocessing_record,
    "constants": {'contractVersion': 'numerical-methods-batch-4-v1', 'parameterOrder': ['variance', 'skewness', 'curtosis', 'entropy', 'intercept'], 'l2': 0.001, 'maxIterations': 500, 'gradientTolerance': 1e-05, 'relativeLossTolerance': 1e-10, 'parameterStepTolerance': 1e-07, 'validationMinDelta': 1e-07, 'validationPatience': 60, 'armijo': {'initialStep': 32.0, 'c': 0.0001, 'rho': 0.5, 'maxBacktracks': 30, 'minimumStep': 1e-12}, 'runs': {'raw-fixed': {'featureSpace': 'raw', 'method': 'fixed', 'step': 4.0}, 'standardized-too-small': {'featureSpace': 'standardized', 'method': 'fixed', 'step': 0.02}, 'standardized-stable': {'featureSpace': 'standardized', 'method': 'fixed', 'step': 4.0}, 'standardized-too-large': {'featureSpace': 'standardized', 'method': 'fixed', 'step': 32.0}, 'standardized-armijo': {'featureSpace': 'standardized', 'method': 'armijo', 'initialStep': 32.0}}, 'finalThreshold': 0.5, 'baselineC': 1.0416666666666667},
    "extremeLogitCheck": extreme_logit_check,
    "gradientCheck": gradient_check,
    "armijoCheck": armijo_check,
    "terminalFixtures": terminal_fixtures,
    "runOrder": RUN_ORDER,
    "runs": run_summaries,
    "finalSelection": {
        "rule": "mathematical-convergence eligibility, then lowest best validation BCE",
        "selectedRunId": selected_run["runId"],
        "selectedIteration": selected_run["bestValidation"]["iteration"],
        "transientUnstableMinimumCannotWin": True,
    },
    "ownership": "optimization owns the five-run numerical comparison",
}

diagnostic_steps = {
    "raw-fixed": {
        "visible": "validation improves and then degrades while steps remain large",
        "plausibleCause": "raw feature scales make the fixed step poorly conditioned",
        "changeOneVariable": "standardize features while keeping the step at 4.0",
        "expectedNextRun": "smoother descent and mathematical convergence",
    },
    "standardized-too-small": {
        "visible": "loss falls safely but the gradient remains large at the iteration cap",
        "plausibleCause": "the fixed step is too small",
        "changeOneVariable": "increase the fixed step from 0.02 to 4.0",
        "expectedNextRun": "reach the gradient tolerance within 500 updates",
    },
    "standardized-stable": {
        "visible": "training and validation losses settle with a small gradient",
        "plausibleCause": "standardization makes the fixed step usable",
        "changeOneVariable": "replace the fixed step with Armijo from 32.0",
        "expectedNextRun": "reject unsafe trials and converge in fewer accepted updates",
    },
    "standardized-too-large": {
        "visible": "a low transient validation point is followed by deterioration",
        "plausibleCause": "the fixed step overshoots",
        "changeOneVariable": "use Armijo backtracking instead of accepting 32.0",
        "expectedNextRun": "accept 16.0 first and retain convergence eligibility",
    },
    "standardized-armijo": {
        "visible": "the first trial is rejected and the gradient tolerance is reached",
        "plausibleCause": "sufficient decrease adapts the usable step",
        "changeOneVariable": "keep the method and inspect the selected test endpoint",
        "expectedNextRun": "the endpoint agrees closely with the library baseline",
    },
}

diagnostics_summary = {
    "contractVersion": CONTRACT_VERSION,
    "outputId": "banknote-training-diagnostics-summary",
    "datasetSha256": observed_dataset_sha256,
    "constantsSha256": "f759186d0be526e84c80f6930d91981af0d303e8138edcbae4f65e4f3d30f041",
    "runOrder": RUN_ORDER,
    "diagnostics": [
        {"runId": run_id, **diagnostic_steps[run_id], "terminal": run_summaries[run_id]["terminal"]}
        for run_id in RUN_ORDER
    ],
    "selectedRunId": selected_run["runId"],
    "finalReport": final_report,
    "baseline": baseline_report,
    "comparison": baseline_comparison,
    "ownership": "training-diagnostics owns cross-run reading and the compact final report",
}

trace_file = {
    "contractVersion": CONTRACT_VERSION,
    "datasetSha256": observed_dataset_sha256,
    "constantsSha256": "f759186d0be526e84c80f6930d91981af0d303e8138edcbae4f65e4f3d30f041",
    "parameterOrder": PARAMETER_ORDER,
    "runs": runs,
}

write_json("optimization-summary.json", optimization_summary)
write_json("training-diagnostics-summary.json", diagnostics_summary)
write_json("banknote-training-traces.json", trace_file)

csv_header = [
    "contract_version", "run_id", "iteration", "feature_space", "method",
    "train_bce", "validation_bce", "objective", "gradient_norm", "parameter_step_norm",
    "accepted_step_size", "backtrack_count", "relative_objective_change", "is_best_validation",
    "w_variance", "w_skewness", "w_curtosis", "w_entropy", "intercept",
]
with (output_dir / "banknote-training-traces.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle, lineterminator="\n")
    writer.writerow(csv_header)
    for run in runs:
        for point in run["trace"]:
            writer.writerow([
                CONTRACT_VERSION, run["runId"], point["iteration"], run["featureSpace"], run["method"],
                point["trainBce"], point["validationBce"], point["objective"], point["gradientNorm"],
                point["parameterStepNorm"], point["acceptedStepSize"], point["backtrackCount"],
                "" if point["relativeObjectiveChange"] is None else point["relativeObjectiveChange"],
                "true" if point["isBestValidation"] else "false", *point["parameters"],
            ])

print(json.dumps({
    "outputs": ["optimization-summary.json", "training-diagnostics-summary.json", "banknote-training-traces.json", "banknote-training-traces.csv"],
    "runRows": {run["runId"]: len(run["trace"]) for run in runs},
    "selectedRunId": selected_run["runId"],
}, ensure_ascii=False, sort_keys=True))

{"outputs": ["optimization-summary.json", "training-diagnostics-summary.json", "banknote-training-traces.json", "banknote-training-traces.csv"], "runRows": {"raw-fixed": 113, "standardized-armijo": 49, "standardized-stable": 485, "standardized-too-large": 74, "standardized-too-small": 501}, "selectedRunId": "standardized-armijo"}


## 7. 读曲线时保留四个问题

1. 你直接看见了什么？
2. 哪个数值原因最合理？
3. 下一次只改变哪个变量？
4. 如果判断正确，下一条轨迹应怎样变化？

`validation-patience` 选择检查点但不证明收敛；`max-iterations`、`non-finite` 与 `line-search-failed` 是安全边界。下一步由浏览器中的确定性 TypeScript 独立复算同一公式与状态机，而不是执行这个 Python Notebook。